# ಪಾಠ 03 - ಏಜೆಂಟಿಕ್ ವಿನ್ಯಾಸ ಮಾದರಿಗಳು

ಈ ಪಾಠದಲ್ಲಿ, ಪರಿಣಾಮಕಾರಿಯಾದ AI ಏಜೆಂಟ್‌ಗಳನ್ನು ನಿರ್ಮಿಸಲು ಮೂರು ಮೂಲಭೂತ ವಿನ್ಯಾಸ ಮಾದರಿಗಳನ್ನು ನಾವು ಅನ್ವೇಷಿಸುತ್ತೇವೆ:

1. **ಸ್ಪಷ್ಟ ಏಜೆಂಟ್ ಸೂಚನೆಗಳು** — ಏಜೆಂಟ್ ವರ್ತನೆಗೆ ಮಾರ್ಗದರ್ಶನ ನೀಡುವ ನಿಖರ, ಪಾತ್ರ-ನಿರ್ಧಾರಕ ಪ್ರಾಂಪ್ಟ್‌ಗಳನ್ನು ರಚಿಸುವುದು  
2. **ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳೊಂದಿಗೆ ರಚನೆಗೊಳ್ಳದ_Output** — ಏಜೆಂಟ್‌ಗಳು ನಿರೀಕ್ಷಿತ, ಮಾನ್ಯತಾ ಪಡೆದ ಡೇಟಾವನ್ನು ಹಿಂತಿರುಗಿಸುವುದನ್ನು ಖಚಿತಪಡಿಸುವುದು  
3. **ಏಕಾಂಗಿ ಹೊಣೆಗಾರಿಕೆ ಏಜೆಂಟ್‌ಗಳು** — ಪ್ರತಿ ಏಜೆಂಟ್ ಒಬ್ಬ ಕೆಲಸವನ್ನು ಚೆನ್ನಾಗಿ ಮಾಡುವಂತೆ ವಿನ್ಯಾಸ ಮಾಡುವುದು  

ನಾವು ಪ್ರತಿ ಮಾದರಿಯನ್ನು **ಪ್ರಯಾಣ ಗಮ್ಯಸ್ಥಳ ಶಿಫಾರಸು ಮಾಡುವ ಘಟಕ** ಸಂದರ್ಭದಲ್ಲಿ ಅನ್ವಯಿಸುವೆವು, ಕ್ರಮೇಣ ಗಮ್ಯಸ್ಥಳಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡುವ, ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸುವ ಮತ್ತು ಲಾಜಿಸ್ಟಿಕ್ಸ್ ನಿರ್ವಹಿಸುವ ವ್ಯವಸ್ಥೆಯನ್ನು ನಿರ್ಮಿಸುತ್ತೇವೆ.


## ಸೆಟ್‌ಅಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ಪ್ಯಾಟರ್ನ್ 1: ಸ್ಪಷ್ಟ ಏಜೆಂಟ್ ಸೂಚನೆಗಳು

ಅತಿ ಪರಿಣಾಮಕಾರಿ ಪ್ಯಾಟರ್ನ್ ಕೂಡ ಅತ್ಯಂತ ಸರಳವಾಗಿದೆ: ನಿಮ್ಮ ಏಜೆಂಟ್‌ಗಾಗಿ ಸ್ಪಷ್ಟ, ವಿವರವಾದ ಸೂಚನೆಗಳನ್ನು ಬರೆಯುವುದು.

ನಂದಿ ಸೂಚನೆಗಳು ಈ ಕೆಳಗಿನವುಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತವೆ:
- **ಯಾರು** ಆಗೆಂಟ್ ಆಗಿದ್ದಾರೆ (ವ್ಯಕ್ತಿತ್ವ ಮತ್ತು ಶೈಲಿ)
- **ಏನು** ಅದು ಮಾಡಬೇಕು (ಹಂತ ಹಂತದ ಹೊಣೆಗಾರಿಕೆಗಳು)
- **ಅದ್ದೇ** ಅದು ಹೇಗೆ ನಡೆದುಕೊಳ್ಳಬೇಕು (ನಿಬಂಧನೆಗಳು ಮತ್ತು ಶೈಲಿ)

ಕೆಳಗಿನಂತೆ, ನಾವು ಸ್ಪಷ್ಟ ಸೂಚನೆಗಳೊಂದಿಗೆ ಟ್ರಾವಲ್ ಕನ್ಸಿಯರ್ಜ್ ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುತ್ತೇವೆ, ಇದು ಅದು ನೀಡುವ ಪ್ರತಿ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನೂ ರೂಪಿಸುತ್ತದೆ.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

##.Pattern 2: Pydantic ಮಾದರಿಗಳೊಂದಿಗೆ ರಚನೆಗೊಳಿಸಿದ output

ಸ್ವತಂತ್ರ ಪಠ್ಯ ಸಂಭಾಷಣೆಗೆ ಉಪಯುಕ್ತವಾಗಿದೆ, ಆದರೆ ಡවුನ್ಸ್ಟ್ರೀಮ್ ವ್ಯವಸ್ಥೆಗಳು ರಚನೆಗೊಳಿಸಿದ ಡೇಟಾವನ್ನು ಅಗತ್ಯವಿರುತ್ತದೆ.
**Pydantic ಮಾದರಿಗಳನ್ನು** **ಟೂಲ್ ಫಂಕ್ಷನ್** ಜೊತೆಗೆ ಜೋಡಿಸುವ ಮೂಲಕ, ನಾವು:

- ಏಜೆಂಟ್ output ಗಾಗಿ ನಿಖರವಾದ ಯೋಜನೆಯನ್ನು ವಿವರಿಸಬಹುದು
- ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಪರಿಶೀಲಿಸಬಹುದು
- ಏಜೆಂಟ್ ಫಲಿತಾಂಶಗಳನ್ನು ಅಪ್ಲಿಕೇಶನ್ ಲಾಜಿಕ್ ನಲ್ಲಿ ವಿಶ್ವಾಸಾರ್ಹವಾಗಿ ಸಂಯೋಜಿಸಬಹುದು

ನಾವು ಏಜೆಂಟ್ ತನ್ನ ಶಿಫಾರಸುಗಳನ್ನು ನಿಜವಾದ ಡೇಟಾದಲ್ಲಿ ನೆಲೆಸಲು ಸ್ಥಳ ಗಮ್ಯಸ್ಥಾನದ ವಿವರಗಳನ್ನು ನೀಡುವ ಟೂಲ್ ಅನ್ನು ಪರಿಚಯಿಸುತ್ತೇವೆ.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## ಪ್ಯಾಟರ್ನ್ 3: ಏಕ ಜವಾಬ್ದಾರಿ ಏಜೆಂಟ್ಗಳು

ಸಂಕೀರ್ಣ ಕಾರ್ಯಗಳನ್ನು ಹಲವು ಕೇಂದ್ರಿತ ಏಜೆಂಟ್ಗಳಲ್ಲಿ ವಿಭಾಗಿಸುವುದು ಲಾಭದಾಯಕ, ಪ್ರತಿಯೊಬ್ಬರಿಗೂ ಒಂದೊಂದು ಜವಾಬ್ದಾರಿ ಇರುತ್ತದೆ:

- ಸ್ಥಳಗಳು ಮತ್ತು ಲಭ್ಯತೆಗಳ ಬಗ್ಗೆ ಗೊತ್ತಿರುವ **ಗಮ್ಯಸ್ಥಳ ತಜ್ಞ**
- ವಿಮಾನಗಳು, ಹೋಟೆಲುಗಳು ಮತ್ತು ಯಾತ್ರೆ ಪ್ರణಾಳಿಕೆಗಳನ್ನು ನಿರ್ವಹಿಸುವ **ಲಾಗಿಸ್ಟಿಕ್ಸ್ ಯೋಜಕ**

ಇದು ಸಾಫ್ಟ್‌ವೇರ್ ಎಂಜಿನಿಯರಿಂಗ್ ತತ್ವವಾದ *ಚಿಂತನೆಗಳ ವಿಭಜನೆ* ಅನ್ನು ಪ್ರತಿಬಿಂಬಿಸುತ್ತದೆ — ಪ್ರತಿಯೊಬ್ಬ ಏಜೆಂಟ್ ಸ್ವತந்திரವಾಗಿ ಪರೀಕ್ಷಿಸಲು, ನಿರ್ವಹಿಸಲು ಮತ್ತು ಸುಧಾರಿಸಲು ಸುಲಭವಾಗಿರುತ್ತವೆ.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನಾವು ಮೂರೂ ಏಜೆಂಟಿಕ್ ವಿನ್ಯಾಸ ಮಾದರಿಗಳನ್ನು ಪ್ರಯಾಣ ಶಿಫಾರಸು ಮಾಡುವ ಸನ್ನಿವೇಶಕ್ಕೆ ಅನ್ವಯಿಸಿಕೊಂಡೆವು:

| ಮಾದರಿ | ಮುಖ್ಯ ಕಲ್ಪನೆ | ಲಾಭ |
|---|---|---|
| **ಸ್ಪಷ್ಟ ಸೂಚನೆಗಳು** | ವ್ಯಕ್ತಿತ್ವ, ಜವಾಬ್ದಾರಿಗಳು ಮತ್ತು ನಿಯಮಗಳನ್ನು ಮುಂಚಿತವಾಗಿ ವ್ಯಾಖ್ಯಾನಿಸು | ಸತತ, ಬ್ರ್ಯಾಂಡ್‌ಗೆ ಅನುಗುಣವಾದ ಏಜೆಂಟ್ ವರ್ತನೆ |
| **ರೂಪರೇಖಿತ output** | ಪ್ರತಿಕ್ರಿಯೆ ಸ್ವರೂಪವಾಗಿ Pydantic ಮಾದರಿಗಳನ್ನು ಬಳಸು | ಪರಿಶೀಲಿಸಲಾದ, ಯಂತ್ರ ಓದುವೋಗ್ಯ ಫಲಿತಾಂಶಗಳು |
| **ಒಂದು ಜವಾಬ್ದಾರಿ** | ಪ್ರತಿ ಏಜೆಂಟ್‌ಗೆ ಒಂದು ಗುರಿ ನಿಗದಿಪಡಿಸಿ | ಸುಲಭವಾಗಿ ಪರೀಕ್ಷಿಸುವುದು, ನಿರ್ವಹಿಸುವುದು ಮತ್ತು ಸಂಯೋಜಿಸುವುದು |

ಈ ಮಾದರಿಗಳು ಸ್ವಾಭಾವಿಕವಾಗಿ ಸಂಯೋಜನವಾಗುತ್ತವೆ — ನೀವು ಸ್ಪಷ್ಟ ಸೂಚನೆಗಳನ್ನು ರೂಪರೇಖಿತ output ಜೊತೆ ಒಂದೇ ಜವಾಬ್ದಾರಿ ಏಜೆಂಟ್ ಒಳಗೆ ಸೇರಿಸಿ ಬಲವಾದ, ಉತ್ಲಾಸಿತ-ಸಿದ್ಧ ವ್ಯವಸ್ಥೆಗಳನ್ನು ನಿರ್ಮಿಸಲು পারেন.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
